# Making chloropleth maps in Altair

Here's a quick example of how to make a chloropleth map in Altair.  In this example, we'll work with a fairly large data set of baby names in France from 1900-2019, broken down by department.

To work with geographical data, we'll use the `geopandas`, which loads `pandas` dataframes, but with support for geographical outlines in the `geojson` format.  You can use these dataframes just as you would a regular `pandas` dataframe, but they will include that extra geographical outline data.

To get started, we'll need to import our libraries.

In [18]:
import altair as alt
import pandas as pd
import geopandas as gpd # Requires geopandas -- e.g.: conda install -c conda-forge geopandas
alt.data_transformers.enable('json') # Let Altair/Vega-Lite work with large data sets

pass

# Reading our names data

Now, let's read in our dataset.  The exported data is in CSV format, but with a `;` separator instead of commas.  The INSEE data collapses rare names or where department-level information has been elided (presumably to protect individuals with uncommon names or who were one of the only ones born with that name in a given year).  We'll strip those out.

In [ ]:
data_dir="../data/"

In [ ]:
names = pd.read_csv(f"{data_dir}dpt2020.csv", sep=";")
names.drop(names[names.preusuel == '_PRENOMS_RARES'].index, inplace=True)
names.drop(names[names.dpt == 'XX'].index, inplace=True)

names.sample(5)

,sexe,preusuel,annais,dpt,nombre
187086,1,AUGUSTIN,2020,08,4
2989503,2,MAGALI,1979,57,57
487659,1,ETIENNE,1967,89,4
319685,1,CLEMENT,2010,43,3
2072831,2,CASSANDRA,1991,80,7


# Loading map data

Next, let's load some map data of regions in France using `geopandas`.  These map data come from the [INSEE] and [IGN] and were processed into the `geojson` format we'll need to work with by [Grégoire David].  Here's the [github] repository.

In this example, we'll work with the simplified departments tiles for the Hexagon, but that repository contains higher-resolution versions, the DOM-TOM, and more.

[Grégoire David]: https://gregoiredavid.fr
[INSEE]: http://www.insee.fr/fr/methodes/nomenclatures/cog/telechargement.asp
[IGN]: https://geoservices.ign.fr/adminexpress
[github]: https://github.com/gregoiredavid/france-geojson/

In [24]:
depts = gpd.read_file('departements-version-simplifiee.geojson')

depts.sample(5)

,code,nom,geometry
39,39,Jura,"POLYGON ((5.51854 47.30419, 5.53165 47.28437, ..."
31,31,Haute-Garonne,"POLYGON ((0.95398 43.78737, 0.97780 43.78644, ..."
91,91,Essonne,"POLYGON ((2.22656 48.77610, 2.23298 48.76620, ..."
68,68,Haut-Rhin,"POLYGON ((7.19828 48.31047, 7.24173 48.30243, ..."
72,72,Sarthe,"POLYGON ((-0.05453 48.38200, -0.04463 48.37976..."


Notice how `depts` is a geopandas dataframe.  We'll use it just as a regular `pandas` dataframe, but it includes the geometry info we need to be able to draw those regions when we pass them into Altair.  We just need to make sure that when we work with our data, we keep them in a geopandas dataframe and not a plain dataframe if we want to draw the departments.

In the next cell, notice how we do a right-merge to bring in department data into names.  We do this as a merge on `depts` because we need a geopandas dataframe.  Remember, `depts` is a geopandas dataframe, while `names` is a regular dataframe.  If we did a left merge on `names`, we'd end up with a regular pandas dataframe. After this merge, both `names` and `depts` will be geopandas dataframes.

**Hint:** Be careful when you do your data joins here.  It's easy to accidentally merge the wrong way to accidentally create a _much bigger_ dataset.

In [25]:
# Keep a reference around to the plain pandas dataframe, without geometry data, just in case
just_names = names

names = depts.merge(names, how='right', left_on='code', right_on='dpt')

names.sample(5)

,code,nom,geometry,sexe,preusuel,annais,dpt,nombre
3488809,75,Paris,"POLYGON ((2.41634 48.84924, 2.46226 48.84254, ...",2,SHIREL,2002,75,64
978396,27,Eure,"POLYGON ((0.29722 49.42986, 0.33898 49.44093, ...",1,LILIAN,2003,27,7
315519,49,Maine-et-Loire,"POLYGON ((-1.24588 47.77672, -1.23825 47.80999...",1,COLIN,2015,49,5
3394297,74,Haute-Savoie,"POLYGON ((6.80252 45.77837, 6.75551 45.76635, ...",2,RENÉE,1920,74,41
2115311,94,Val-de-Marne,"POLYGON ((2.33190 48.81701, 2.36395 48.81632, ...",2,CHRISTINE,2018,94,4


# Show a name over all years

Now we'll choose a name to show across all years.  To that, we'll group all of the names in a department together (squashing the years together) and use the sum.

In [28]:
grouped = names.groupby(['dpt', 'preusuel', 'sexe'], as_index=False).sum()
grouped = depts.merge(grouped, how='right', left_on='code', right_on='dpt') # Add geometry data back in
grouped

,code,nom,geometry,dpt,preusuel,sexe,nombre
0,01,Ain,"POLYGON ((4.78021 46.17668, 4.79458 46.21832, ...",01,AARON,1,160
1,01,Ain,"POLYGON ((4.78021 46.17668, 4.79458 46.21832, ...",01,ABBY,2,3
2,01,Ain,"POLYGON ((4.78021 46.17668, 4.79458 46.21832, ...",01,ABDALLAH,1,7
3,01,Ain,"POLYGON ((4.78021 46.17668, 4.79458 46.21832, ...",01,ABDEL,1,3
4,01,Ain,"POLYGON ((4.78021 46.17668, 4.79458 46.21832, ...",01,ABDELKADER,1,3
...,...,...,...,...,...,...,...
239574,NaN,NaN,None,974,ÉSAÏE,1,3
239575,NaN,NaN,None,974,ÉTHAN,1,53
239576,NaN,NaN,None,974,ÉTIENNE,1,3
239577,NaN,NaN,None,974,ÉVA,2,32


Now let's pick a name and check out how it's distribution over the last 120 years across Metropolitan France.  In this example, I choose the name “Lucien,” which I rather like for some reason.

In [31]:
name = 'LUCIEN'
subset = grouped[grouped.preusuel == name]
alt.Chart(subset).mark_geoshape(stroke='white').encode(
    tooltip=['nom', 'code', 'nombre'],
    color='nombre',
).properties(width=800, height=600)

alt.Chart(...)

## Multi-line chart: gender trends per name over time

This section builds the visualization style shown in your screenshot: one mini line chart per name, with male and female trends plotted on the same axes over time.

In [ ]:
import altair as alt
import pandas as pd

# Load and prepare name data for trend lines
trend = pd.read_csv("../data/dpt2020.csv", sep=";")
trend = trend[(trend["preusuel"] != "_PRENOMS_RARES") & (trend["dpt"] != "XX")].copy()

trend["annais"] = trend["annais"].astype(int)
trend["nombre"] = trend["nombre"].astype(int)
trend["gender"] = trend["sexe"].map({1: "Male", 2: "Female"})


selected_names = ["DOMINIQUE", "CAMILLE", "CLAUDE", "ALIX", "CHARLIE", "EDEN"]
trend = trend[trend["preusuel"].isin(selected_names)]

# Aggregate across all departments -> yearly counts per name and gender
trend_year = (
    trend.groupby(["annais", "preusuel", "gender"], as_index=False)["nombre"]
    .sum()
    .rename(columns={"annais": "year", "preusuel": "name", "nombre": "births"})
)

trend_year.head()

,year,name,gender,births
0,1900,ALIX,Female,47
1,1900,ALIX,Male,6
2,1900,CAMILLE,Female,711
3,1900,CAMILLE,Male,1188
4,1900,CLAUDE,Male,626


In [ ]:
color_scale = alt.Scale(domain=["Male", "Female"], range=["#4C66AF", "#C4544B"])
name_order = ["DOMINIQUE", "CAMILLE", "CLAUDE", "ALIX", "CHARLIE", "EDEN"]

multi_line = (
    alt.Chart(trend_year)
    .mark_line(strokeWidth=2)
    .encode(
        x=alt.X("year:Q", title=None, axis=alt.Axis(format="d", tickCount=4, grid=False)),
        y=alt.Y("births:Q", title="Births / year", axis=alt.Axis(grid=False)),
        color=alt.Color("gender:N", scale=color_scale, legend=alt.Legend(orient="top", title=None)),
        tooltip=[
            alt.Tooltip("name:N", title="Name"),
            alt.Tooltip("gender:N", title="Gender"),
            alt.Tooltip("year:Q", title="Year", format="d"),
            alt.Tooltip("births:Q", title="Births", format=","),
        ],
    )
    .properties(width=180, height=140)
    .facet(
        facet=alt.Facet("name:N", sort=name_order, header=alt.Header(title=None, labelFontSize=18, labelPadding=8)),
        columns=3,
    )
    .resolve_scale(y="shared")
    .properties(title="Multi-line chart: gender trends per name over time")
)

multi_line

alt.FacetChart(...)